### Automated Order Data Cleanup and Insight Generation with Pandas & SQL

In [4]:
# Used data direct from the Kaggle
import kaggle
#!kaggle datasets download varshaverma/retail-orders -f orders.csv

In [ ]:
# If you want to take the dataset from the zipfile
import zipfile
zip_ref=zipfile.ZipExtFile('archive (12).zip')
zip_ref.extractall()
zip_ref.close()

In [6]:
import pandas as pd
df=pd.read_csv('orders.csv')
df.head(5)

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5


In [7]:
# Findouut the unique value from the Ship Mode
df['Ship Mode'].unique()

array(['Second Class', 'Standard Class', 'Not Available', 'unknown',
       'First Class', nan, 'Same Day'], dtype=object)

In [8]:
# Convert the columne name in lowercase 
df.columns=df.columns.str.lower()

In [9]:
# Remove the space with underscore
df.columns=df.columns.str.replace(' ','_')

In [10]:
df.head(2)

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3


In [11]:
# Discount
df['discount']=df['list_price']*df['discount_percent']*.01

In [12]:
# Sale price
df['sale_price']=df['list_price']-df['discount']

In [13]:
# Profit
df['profit']=df['sale_price']-df['cost_price']

In [14]:
# Ship Mode contain the Not Available and Unknown value these value replaced by the nan
import numpy as np
df['ship_mode']=df['ship_mode'].replace(['Not Available','Unknown'],np.nan)

In [15]:
df.head(8)

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent,discount,sale_price,profit
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2,5.2,254.8,14.8
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3,21.9,708.1,108.1
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5,0.5,9.5,-0.5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2,19.2,940.8,160.8
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5,1.0,19.0,-1.0
5,6,2022-03-13,NaN,Consumer,United States,Los Angeles,California,90032,West,Furniture,Furnishings,FUR-FU-10001487,50,50,7,3,1.5,48.5,-1.5
6,7,2022-12-28,Standard Class,Consumer,United States,Los Angeles,California,90032,West,Office Supplies,Art,OFF-AR-10002833,10,10,4,3,0.3,9.7,-0.3
7,8,2022-01-25,Standard Class,Consumer,United States,Los Angeles,California,90032,West,Technology,Phones,TEC-PH-10002275,860,910,6,5,45.5,864.5,4.5


In [16]:
df.dtypes

order_id              int64
order_date           object
ship_mode            object
segment              object
country              object
city                 object
state                object
postal_code           int64
region               object
category             object
sub_category         object
product_id           object
cost_price            int64
list_price            int64
quantity              int64
discount_percent      int64
discount            float64
sale_price          float64
profit              float64
dtype: object

In [17]:
# Convert the datetime colm in datetime data type

df['order_date']=pd.to_datetime(df['order_date'],format="%Y-%m-%d")

In [18]:
# Drop list price, discount percent,cost price

df.drop(columns=['cost_price','list_price','discount_percent'],inplace=True)

In [19]:
df.columns

Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'quantity', 'discount', 'sale_price', 'profit'],
      dtype='object')

In [20]:
# Here, it is establish the connection with sql server 
import sqlalchemy as sal
engine=sal.create_engine('mssql://Varshaverma\SQLEXPRESS/mysql?driver=ODBC+DRIVER+17+FOR+SQL+SERVER')

In [21]:
conn = engine.connect()

In [ ]:
# Load the data in sql server
df.to_sql('df_orders',con=conn,index=False, if_exists='append')